In [85]:
import pandas as pd
df=pd.read_csv('dataforbotescalationsystem.csv')
print(df.head())

  conversation_id                                           messages  label  \
0        conv_001  User: Hi, did my refund go through? Order #773...      0   
1        conv_002  User: My package was delivered to the wrong ad...      1   
2        conv_003  User: I've been waiting 2 weeks for my refund....      1   
3        conv_004  User: My login stopped working after your upda...      1   
4        conv_005  User: Hi, I ordered 3 items but only got 2. Sh...      0   

                           reasons  severity  
0             resolved,smooth_flow      0.52  
1             frustration,fallback      0.72  
2  frustration,repetition,fallback      0.89  
3  frustration,repetition,fallback      0.89  
4               resolved,clarified      0.54  


In [86]:
df.dropna(inplace=True)
df['messages'] = df['messages'].str.strip()
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 230 entries, 0 to 229
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   conversation_id  230 non-null    str    
 1   messages         230 non-null    str    
 2   label            230 non-null    int64  
 3   reasons          230 non-null    str    
 4   severity         230 non-null    float64
dtypes: float64(1), int64(1), str(3)
memory usage: 9.1 KB
None


In [87]:
def parse_conversation(text):
    turns = text.split("||")
    parsed = []
    
    for t in turns:
        if ":" in t:
            role, msg = t.split(":", 1)
            parsed.append((role.strip(), msg.strip()))
    
    return parsed

df['parsed'] = df['messages'].apply(parse_conversation)
print(df['parsed'].iloc[0])

[('User', 'Hi, did my refund go through? Order #7734.'), ('Bot', 'Let me check… Yes, a refund of Rs.89.99 was processed on April 2nd. It should appear on your statement by April 7th.'), ('User', "Okay cool. I'll keep an eye out. Thanks."), ('Bot', "You're welcome! Feel free to reach out if it doesn't show up.")]


In [88]:
def split_roles(parsed):
    user_msgs = [m for r, m in parsed if r == "User"]
    bot_msgs = [m for r, m in parsed if r == "Bot"]
    return user_msgs, bot_msgs

df['user_msgs'], df['bot_msgs'] = zip(*df['parsed'].apply(split_roles))


In [89]:
df['num_turns'] = df['parsed'].apply(len)
df['num_user_msgs'] = df['user_msgs'].apply(len)
df['num_bot_msgs'] = df['bot_msgs'].apply(len)
df['user_bot_ratio'] = df['num_user_msgs'] / (df['num_bot_msgs'] + 1e-5)
print(df[['num_turns', 'num_user_msgs', 'num_bot_msgs', 'user_bot_ratio']].head())

   num_turns  num_user_msgs  num_bot_msgs  user_bot_ratio
0          4              2             2        0.999995
1          9              5             4        1.249997
2          7              4             3        1.333329
3          8              4             4        0.999998
4          7              4             3        1.333329


In [90]:
print(df.groupby('label')['num_turns'].mean())

label
0    5.217391
1    8.184783
Name: num_turns, dtype: float64


In [91]:
import nltk
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [92]:
from nltk.sentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

def sentiment_scores(messages):
    return [sia.polarity_scores(m)['compound'] for m in messages]

df['user_sentiments'] = df['user_msgs'].apply(sentiment_scores)

In [93]:
print(df['user_msgs'].iloc[0])
print(df['user_sentiments'].iloc[8])

['Hi, did my refund go through? Order #7734.', "Okay cool. I'll keep an eye out. Thanks."]
[0.0, 0.1027, 0.0772, -0.1139]


In [94]:
df['avg_sentiment'] = df['user_sentiments'].apply(
    lambda x: sum(x)/len(x) if len(x) > 0 else 0
)
df['min_sentiment'] = df['user_sentiments'].apply(
    lambda x: min(x) if len(x) > 0 else 0
)
def sentiment_trend(scores):
    if len(scores) < 2:
        return 0
    return scores[-1] - scores[0]

df['sentiment_trend'] = df['user_sentiments'].apply(sentiment_trend)

In [95]:
print(df.groupby('label')['avg_sentiment'].mean())
print(df.groupby('label')['min_sentiment'].mean())

label
0    0.187416
1   -0.116296
Name: avg_sentiment, dtype: float64
label
0   -0.101357
1   -0.410716
Name: min_sentiment, dtype: float64


In [96]:
import numpy as np
import sklearn
print("Installed successfully")
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def repetition_score(user_msgs):
    if len(user_msgs) < 2:
        return 0
    
    vectorizer = TfidfVectorizer().fit(user_msgs)
    vectors = vectorizer.transform(user_msgs)
    
    sim_scores = []
    
    for i in range(1, len(user_msgs)):
        sim = cosine_similarity(vectors[i], vectors[i-1])[0][0]
        sim_scores.append(sim)
    
    return max(sim_scores)  # strongest repetition

df['repetition_score'] = df['user_msgs'].apply(repetition_score)

Installed successfully


In [97]:
fallback_phrases = [
    "i don't understand",
    "sorry",
    "can you rephrase",
    "i didn't get that",
    "unable to help",
    "not sure",
    "cannot help",
    "please clarify"
]
def fallback_count(bot_msgs):
    count = 0
    
    for msg in bot_msgs:
        msg_lower = msg.lower()
        if any(p in msg_lower for p in fallback_phrases):
            count += 1
    
    return count

df['fallback_count'] = df['bot_msgs'].apply(fallback_count)

In [98]:
print(df.groupby('label')['fallback_count'].mean())

label
0    0.326087
1    0.836957
Name: fallback_count, dtype: float64


In [99]:
frustration_keywords = [
    "frustrated", "angry", "not working", "still",
    "again", "why", "worst", "hate", "useless",
    "issue", "problem", "error"
]

def frustration_score(user_msgs):
    score = 0
    
    for msg in user_msgs:
        msg_lower = msg.lower()
        if any(k in msg_lower for k in frustration_keywords):
            score += 1
    
    return score

df['frustration_score'] = df['user_msgs'].apply(frustration_score)

In [100]:
generic_responses = [
    "i am here to help",
    "please clarify",
    "can you provide more details",
    "let me check",
    "i will assist you",
    "thanks for reaching out"
]

def generic_ratio(bot_msgs):
    if len(bot_msgs) == 0:
        return 0
    
    generic_count = 0
    
    for msg in bot_msgs:
        msg_lower = msg.lower()
        if any(g in msg_lower for g in generic_responses):
            generic_count += 1
    
    return generic_count / len(bot_msgs)

df['generic_response_ratio'] = df['bot_msgs'].apply(generic_ratio)


In [116]:
features = [
    'num_turns',
    'num_user_msgs',
    'num_bot_msgs',
    'user_bot_ratio',
    'avg_sentiment',
    'min_sentiment',
    'sentiment_trend',
    'fallback_count',
    'frustration_score',
    'generic_response_ratio'
]

X = df[features]
y = df['label']

print (X.head())
print(y.head())


   num_turns  num_user_msgs  num_bot_msgs  user_bot_ratio  avg_sentiment  \
0          4              2             2        0.999995       0.363450   
1          9              5             4        1.249997      -0.341840   
2          7              4             3        1.333329      -0.361425   
3          8              4             4        0.999998       0.028425   
4          7              4             3        1.333329       0.252250   

   min_sentiment  sentiment_trend  fallback_count  frustration_score  \
0         0.0000           0.7269               0                  0   
1        -0.6093           0.0998               1                  0   
2        -0.7233          -0.3621               1                  2   
3        -0.2263           0.5663               1                  1   
4         0.0000           0.3612               1                  0   

   generic_response_ratio  
0                0.500000  
1                0.000000  
2                0.000000 

In [183]:
# Boost signals directly in existing features
X['fallback_count'] = X['fallback_count'] * 1.5  # amplify bot failures
X['frustration_score'] = X['frustration_score'] * 1.5  # amplify frustration
X['generic_response_ratio'] = X['generic_response_ratio'] * 1.2  # slightly amplify generic responses
X['sentiment_trend'] = X['sentiment_trend'].apply(lambda x: -x if x < 0 else 0)  # treat drops as positive signal

In [185]:
# Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [186]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.9347826086956522
              precision    recall  f1-score   support

           0       1.00      0.89      0.94        28
           1       0.86      1.00      0.92        18

    accuracy                           0.93        46
   macro avg       0.93      0.95      0.93        46
weighted avg       0.94      0.93      0.94        46



In [187]:
print(hasattr(model, "coef_"))

True


In [188]:
print(df.groupby('label').mean(numeric_only=True))

       severity  num_turns  num_user_msgs  num_bot_msgs  user_bot_ratio  \
label                                                                     
0      0.552971   5.217391       2.717391      2.500000        1.084536   
1      0.816522   8.184783       4.510870      3.673913        1.236410   

       avg_sentiment  min_sentiment  sentiment_trend  repetition_score  \
label                                                                    
0           0.187416      -0.101357         0.528997          0.054889   
1          -0.116296      -0.410716         0.094720          0.144745   

       fallback_count  frustration_score  generic_response_ratio  
label                                                             
0            0.326087           0.159420                0.054348  
1            0.836957           0.793478                0.038043  


In [194]:
def predict_escalation(conversation_text, model, scaler, threshold=0.5):
    
    # --- Parse ---
    parsed = parse_conversation(conversation_text)
    user_msgs, bot_msgs = split_roles(parsed)
    
    # --- Feature computation ---
    num_turns = len(parsed)
    num_user_msgs = len(user_msgs)
    num_bot_msgs = len(bot_msgs)
    user_bot_ratio = num_user_msgs / (num_bot_msgs + 1e-5)
    
    # Sentiment
    sentiments = sentiment_scores(user_msgs)
    avg_sentiment = sum(sentiments)/len(sentiments) if sentiments else 0
    min_sentiment = min(sentiments) if sentiments else 0
    sentiment_trend = sentiments[-1] - sentiments[0] if len(sentiments) > 1 else 0
    
    # Repetition
    rep_score = repetition_score(user_msgs)
    
    # Fallback
    fb_count = fallback_count(bot_msgs)
    
    # Frustration
    fr_score = frustration_score(user_msgs)
    
    # Generic responses
    gen_ratio = generic_ratio(bot_msgs)
    
    # --- Feature vector ---
    features = [[
        num_turns,
        num_user_msgs,
        num_bot_msgs,
        user_bot_ratio,
        avg_sentiment,
        min_sentiment,
        sentiment_trend,
        fb_count,
        fr_score,
        gen_ratio
    ]]
    
    # --- Scale ---
    features_scaled = scaler.transform(features)
    
    # --- Predict probability ---
    prob = model.predict_proba(features_scaled)[0][1]
    
    # --- Decision ---
    decision = 1 if prob >= threshold else 0
    
    return prob, decision

In [195]:
print(hasattr(model, "coef_"))

True


In [196]:
conv1 = """User: Hi || Bot: Hello || User: I need help with login || Bot: Sure, reset your password || User: Thanks || Bot: You're welcome"""

print(predict_escalation(conv1, model, scaler))

(np.float64(0.08881646271528519), 0)


C:\Users\hp\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [209]:
conv=""" "User: Hey there! How’s it going? || "
    "Bot: Hello! I’m good, how about you? || "
    "User: I’m fine, thanks for asking || "
    "Bot: Glad to hear that!"""

In [210]:
print(predict_escalation(conv, model, scaler))

(np.float64(1.500474336195285e-07), 0)


C:\Users\hp\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
